# Day 11 — Gold Full / Incremental Load v3

Builds all Gold dim and fact tables with proper full and incremental load support.

| Widget | Values | Default |
|---|---|---|
| `load_type` | `full` \| `incremental` | `full` |
| `pipeline_id` | any string label | `pl_gold_v3` |
| `silver_base` | path to silver-volume | `/Volumes/dbw_ev_intelligence_dev/default/silver-volume` |
| `gold_base` | path to gold-volume | `/Volumes/dbw_ev_intelligence_dev/default/gold-volume` |

**Full load:** deletes existing Gold table, rewrites from scratch with `overwriteSchema=true`.

**Incremental load:** Delta MERGE upsert on natural key for dims, partition overwrite by year/month for facts.

| Layer | Full | Incremental |
|---|---|---|
| Dims (SCD1) | overwrite | MERGE on natural key |
| Dims (SCD2) | overwrite | MERGE — expire old row, insert new row |
| Facts | overwrite partitioned | MERGE on fact grain key |
| DimTime | overwrite (generated, never changes) | skip if already exists |

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime, timezone
print('Imports OK')

In [ ]:
# Widget params
dbutils.widgets.removeAll()
dbutils.widgets.text('load_type',   'full',        'Load Type (full / incremental)')
dbutils.widgets.text('pipeline_id', 'pl_gold_v3',  'Pipeline ID')
dbutils.widgets.text('silver_base', '/Volumes/dbw_ev_intelligence_dev/default/silver-volume', 'Silver Base Path')
dbutils.widgets.text('gold_base',   '/Volumes/dbw_ev_intelligence_dev/default/gold-volume',   'Gold Base Path')

LOAD_TYPE   = dbutils.widgets.get('load_type').strip().lower()
PIPELINE    = dbutils.widgets.get('pipeline_id').strip()
SILVER_BASE = dbutils.widgets.get('silver_base').strip()
GOLD_BASE   = dbutils.widgets.get('gold_base').strip()

if LOAD_TYPE not in ('full', 'incremental'):
    raise ValueError(f"load_type must be 'full' or 'incremental', got: {LOAD_TYPE!r}")

SILVER_API = f'{SILVER_BASE}/api'
SILVER_RT  = f'{SILVER_BASE}/realtime'
GOLD_DIM   = f'{GOLD_BASE}/dims'
GOLD_FACT  = f'{GOLD_BASE}/facts'
RUN_TS     = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

print(f'load_type   : {LOAD_TYPE}')
print(f'pipeline_id : {PIPELINE}')
print(f'silver_base : {SILVER_BASE}')
print(f'gold_base   : {GOLD_BASE}')
print(f'RUN_TS      : {RUN_TS}')

In [ ]:
# Helpers

def silver(name):
    return spark.read.format('delta').load(f'{SILVER_API}/{name}')

def read_gold(name, base=GOLD_DIM):
    return spark.read.format('delta').load(f'{base}/{name}')

def path_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

def meta(df):
    return (df
        .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
        .withColumn('gold_pipeline',    F.lit(PIPELINE)))

def time_key(ts):
    return F.concat_ws('',
        F.year(ts).cast('string'),
        F.lpad(F.month(ts).cast('string'),      2, '0'),
        F.lpad(F.dayofmonth(ts).cast('string'), 2, '0'),
        F.lpad(F.hour(ts).cast('string'),       2, '0'),
    )

# ── Write helpers ─────────────────────────────────────────────────────────────

def write_dim_scd1(df, name, natural_key):
    """SCD Type 1: full overwrite or MERGE (update-in-place)."""
    path = f'{GOLD_DIM}/{name}'
    if LOAD_TYPE == 'full' or not path_exists(path):
        try:
            dbutils.fs.rm(path, recurse=True)
        except Exception:
            pass
        (df.write.format('delta')
           .mode('overwrite')
           .option('overwriteSchema', 'true')
           .save(path))
    else:
        dt = DeltaTable.forPath(spark, path)
        (dt.alias('t')
           .merge(df.alias('s'), f't.{natural_key} = s.{natural_key}')
           .whenMatchedUpdateAll()
           .whenNotMatchedInsertAll()
           .execute())
    cnt = spark.read.format('delta').load(path).count()
    print(f'  DIM  {name:<25} {cnt:>8} rows  [{LOAD_TYPE}]')

def write_dim_scd2(df, name, natural_key):
    """SCD Type 2: full overwrite or expire-old + insert-new on incremental."""
    path = f'{GOLD_DIM}/{name}'
    if LOAD_TYPE == 'full' or not path_exists(path):
        try:
            dbutils.fs.rm(path, recurse=True)
        except Exception:
            pass
        (df.write.format('delta')
           .mode('overwrite')
           .option('overwriteSchema', 'true')
           .save(path))
    else:
        dt = DeltaTable.forPath(spark, path)
        # Expire current rows that have changed, then insert new versions
        (
            dt.alias('t')
            .merge(
                df.alias('s'),
                f't.{natural_key} = s.{natural_key} AND t.is_current = true'
            )
            # If matched and something changed: expire the old row
            .whenMatchedUpdate(
                condition='t.gold_ingested_at <> s.gold_ingested_at',
                set={
                    'is_current': F.lit(False),
                    'valid_to':   F.lit(RUN_TS).cast('timestamp'),
                }
            )
            .execute()
        )
        # Insert new versions for changed rows
        new_rows = df.join(
            spark.read.format('delta').load(path).filter(F.col('is_current') == True)
                 .select(natural_key),
            on=natural_key, how='left_anti'
        )
        if new_rows.limit(1).count() > 0:
            new_rows.write.format('delta').mode('append').save(path)
    cnt = spark.read.format('delta').load(path).count()
    print(f'  DIM  {name:<25} {cnt:>8} rows  [{LOAD_TYPE}]')

def write_fact(df, name, grain_key, partition_cols=None):
    """Full overwrite or MERGE on grain key for incremental."""
    path = f'{GOLD_FACT}/{name}'
    if LOAD_TYPE == 'full' or not path_exists(path):
        try:
            dbutils.fs.rm(path, recurse=True)
        except Exception:
            pass
        w = (df.write.format('delta')
               .mode('overwrite')
               .option('overwriteSchema', 'true'))
        if partition_cols:
            w = w.partitionBy(*partition_cols)
        w.save(path)
    else:
        dt = DeltaTable.forPath(spark, path)
        (dt.alias('t')
           .merge(df.alias('s'), f't.{grain_key} = s.{grain_key}')
           .whenMatchedUpdateAll()
           .whenNotMatchedInsertAll()
           .execute())
    cnt = spark.read.format('delta').load(path).count()
    print(f'  FACT {name:<25} {cnt:>8} rows  [{LOAD_TYPE}]')

print('Helpers defined')

In [ ]:
# ── DIMENSION TABLES ─────────────────────────────────────────────────────────
print('=' * 55)
print('DIMS')
print('=' * 55)

# DimState — SCD1
write_dim_scd1(meta(silver('states').select(
    F.monotonically_increasing_id().alias('state_key'),
    F.col('state_code'),
    F.trim('name').alias('state_name'),
    F.trim('country').alias('country'),
).filter(F.col('state_code').isNotNull())), 'DimState', 'state_code')

# DimCity — SCD1
write_dim_scd1(meta(silver('cities').select(
    F.monotonically_increasing_id().alias('city_key'),
    F.col('city_id'),
    F.trim('city_name').alias('city_name'),
    F.trim('state_code').alias('state_code'),
    F.col('postcode'),
).filter(F.col('city_id').isNotNull())), 'DimCity', 'city_id')

# DimStation — SCD1
write_dim_scd1(meta(silver('stations').select(
    F.monotonically_increasing_id().alias('station_key'),
    F.col('station_id'),
    F.trim('name').alias('station_name'),
    F.trim('state_code').alias('state_code'),
    F.trim('city').alias('city'),
    F.col('latitude'), F.col('longitude'),
    F.trim('charger_type').alias('charger_type'),
    F.col('max_power_kw'), F.col('num_connectors'),
    F.trim('operator').alias('operator'),
    F.col('is_active'), F.col('commissioned_date'),
).filter(F.col('station_id').isNotNull())), 'DimStation', 'station_id')

# DimCharger — SCD2
write_dim_scd2(meta(silver('chargers').select(
    F.monotonically_increasing_id().alias('charger_key'),
    F.col('charger_id'), F.col('station_id'),
    F.trim('connector_type').alias('connector_type'),
    F.col('max_kw'), F.col('firmware_version'), F.col('install_date'),
    F.trim('status').alias('status'),
    F.col('created_at').alias('valid_from'),
    F.lit(None).cast('timestamp').alias('valid_to'),
    F.lit(True).alias('is_current'),
).filter(F.col('charger_id').isNotNull())), 'DimCharger', 'charger_id')

# DimCustomer — SCD2
write_dim_scd2(meta(silver('customers').select(
    F.monotonically_increasing_id().alias('customer_key'),
    F.col('customer_id'),
    F.trim('full_name').alias('full_name'),
    F.trim('email').alias('email'),
    F.trim('phone').alias('phone'),
    F.trim('loyalty_tier').alias('loyalty_tier'),
    F.col('signup_date'),
    F.col('created_at').alias('valid_from'),
    F.lit(None).cast('timestamp').alias('valid_to'),
    F.lit(True).alias('is_current'),
).filter(F.col('customer_id').isNotNull())), 'DimCustomer', 'customer_id')

# DimVehicle — SCD1
write_dim_scd1(meta(silver('vehicles').select(
    F.monotonically_increasing_id().alias('vehicle_key'),
    F.col('vehicle_id'),
    F.trim('make').alias('make'), F.trim('model').alias('model'),
    F.col('year'), F.col('battery_capacity_kwh'), F.col('range_km'),
    F.trim('vehicle_type').alias('vehicle_type'),
    F.trim('registration_state').alias('registration_state'),
    F.col('partner_id'),
).filter(F.col('vehicle_id').isNotNull())), 'DimVehicle', 'vehicle_id')

# DimEmployee — SCD1
write_dim_scd1(meta(silver('employees').select(
    F.monotonically_increasing_id().alias('employee_key'),
    F.col('employee_id'),
    F.trim('name').alias('full_name'),
    F.trim('department').alias('department'),
    F.col('station_id'),
    F.trim('state').alias('state'),
    F.col('hire_date'),
).filter(F.col('employee_id').isNotNull())), 'DimEmployee', 'employee_id')

# DimPartner — SCD1
write_dim_scd1(meta(silver('partners').select(
    F.monotonically_increasing_id().alias('partner_key'),
    F.col('partner_id'),
    F.trim('partner_name').alias('partner_name'),
    F.trim('state').alias('state'),
    F.trim('status').alias('status'),
    F.col('revenue_share_pct'), F.col('contract_start'), F.col('contract_end'),
).filter(F.col('partner_id').isNotNull())), 'DimPartner', 'partner_id')

# DimChargeCard — SCD1
write_dim_scd1(meta(silver('charge_cards').select(
    F.monotonically_increasing_id().alias('card_key'),
    F.col('card_id'), F.col('customer_id'), F.col('rfid_uid'),
    F.when(
        F.col('card_number').isNotNull() & (F.length('card_number') >= 8),
        F.concat(F.substring('card_number', 1, 4), F.lit('-****-****-'), F.substring('card_number', -4, 4))
    ).otherwise(F.lit('****')).alias('card_number_masked'),
    F.trim('card_type').alias('card_type'),
    F.trim('status').alias('status'),
).filter(F.col('card_id').isNotNull())), 'DimChargeCard', 'card_id')

# DimTariff — SCD2 (is_current comes from source)
write_dim_scd2(meta(silver('tariffs').select(
    F.monotonically_increasing_id().alias('tariff_key'),
    F.col('tariff_id'), F.col('version'),
    F.trim('peak_offpeak').alias('peak_offpeak'),
    F.col('rate_per_kwh'), F.col('effective_from'), F.col('effective_to'),
    F.col('is_current'),
    F.col('created_at').alias('valid_from'),
    F.lit(None).cast('timestamp').alias('valid_to'),
).filter(F.col('tariff_id').isNotNull())), 'DimTariff', 'tariff_id')

# DimTime — generated 2020-2030, skip on incremental if already exists
dim_time_path = f'{GOLD_DIM}/DimTime'
if LOAD_TYPE == 'full' or not path_exists(dim_time_path):
    TOTAL_HOURS = 365 * 24 * 11
    dim_time = meta(spark.range(TOTAL_HOURS).select(
        F.expr("timestamp '2020-01-01 00:00:00' + make_interval(0,0,0,0,cast(id as int),0,0)").alias('ts')
    ).select(
        F.concat_ws('',
            F.year('ts').cast('string'),
            F.lpad(F.month('ts').cast('string'), 2, '0'),
            F.lpad(F.dayofmonth('ts').cast('string'), 2, '0'),
            F.lpad(F.hour('ts').cast('string'), 2, '0'),
        ).alias('time_key'),
        F.col('ts'), F.to_date('ts').alias('date'),
        F.year('ts').alias('year'), F.month('ts').alias('month'),
        F.dayofmonth('ts').alias('day'), F.hour('ts').alias('hour'),
        F.date_format('ts', 'EEEE').alias('day_of_week'),
        F.when(F.dayofweek('ts').isin(1, 7), True).otherwise(False).alias('is_weekend'),
        F.quarter('ts').alias('quarter'),
        F.when(F.month('ts') >= 7,
            F.concat(F.lit('FY'), (F.year('ts') + 1).cast('string'))
        ).otherwise(
            F.concat(F.lit('FY'), F.year('ts').cast('string'))
        ).alias('financial_year_au'),
    ))
    try:
        dbutils.fs.rm(dim_time_path, recurse=True)
    except Exception:
        pass
    (dim_time.write.format('delta')
             .mode('overwrite')
             .option('overwriteSchema', 'true')
             .save(dim_time_path))
    cnt = spark.read.format('delta').load(dim_time_path).count()
    print(f'  DIM  {"DimTime":<25} {cnt:>8} rows  [{LOAD_TYPE}]')
else:
    print(f'  DIM  {"DimTime":<25} skipped (already exists, incremental)')

In [ ]:
# ── LOAD DIM SURROGATE KEY LOOKUPS ───────────────────────────────────────────
lk_station  = read_gold('DimStation').select('station_key',  'station_id')
lk_charger  = read_gold('DimCharger').filter(F.col('is_current') == True).select('charger_key',  'charger_id')
lk_customer = read_gold('DimCustomer').filter(F.col('is_current') == True).select('customer_key', 'customer_id')
lk_vehicle  = read_gold('DimVehicle').select('vehicle_key',  'vehicle_id')
lk_employee = read_gold('DimEmployee').select('employee_key', 'employee_id')
lk_tariff   = read_gold('DimTariff').filter(F.col('is_current') == True).select('tariff_key', 'tariff_id')
print('Dim lookups loaded')

In [ ]:
# ── FACT TABLES ──────────────────────────────────────────────────────────────
print('=' * 55)
print('FACTS')
print('=' * 55)

sessions_raw   = silver('sessions')
payments_raw   = silver('payments')
complaints_raw = silver('complaints')
maintenance_raw = silver('maintenance_events')
rt_sessions    = spark.read.format('delta').load(f'{SILVER_RT}/charging_sessions')

# ── FactChargingSession ───────────────────────────────────────────────────────
payment_lookup = payments_raw.select(
    'payment_id',
    F.col('amount_aud').alias('payment_amount_aud'),
    F.col('gst').alias('payment_gst'),
    F.trim('payment_mode').alias('payment_mode'),
    F.trim('status').alias('payment_status'),
    'processed_at',
)

fact_session = meta(
    sessions_raw
    .join(lk_station,     on='station_id',  how='left')
    .join(lk_customer,    on='customer_id', how='left')
    .join(lk_vehicle,     on='vehicle_id',  how='left')
    .join(payment_lookup, on='payment_id',  how='left')
    .withColumn('session_time_key', time_key(F.col('started_at')))
    .withColumn('session_date',  F.to_date('started_at'))
    .withColumn('session_year',  F.year('started_at'))
    .withColumn('session_month', F.month('started_at'))
    .withColumn('cost_reconciliation',
        F.when(F.col('payment_amount_aud').isNull(), 'UNMATCHED')
        .when(F.col('cost_aud').isNotNull() &
              (F.abs(F.col('payment_amount_aud') - F.col('cost_aud')) < 0.01), 'MATCH')
        .otherwise('MISMATCH'))
    .withColumn('cost_per_kwh',
        F.when(F.col('energy_kwh').isNotNull() & (F.col('energy_kwh') > 0) & F.col('cost_aud').isNotNull(),
               (F.col('cost_aud') / F.col('energy_kwh')).cast('decimal(8,4)')))
    .select(
        'session_id', 'station_key', 'station_id',
        'customer_key', 'customer_id', 'vehicle_key', 'vehicle_id',
        'payment_id', 'session_time_key', 'session_date',
        'session_year', 'session_month',
        F.col('started_at').alias('session_start_ts'),
        F.col('ended_at').alias('session_end_ts'),
        'duration_min', 'energy_kwh', 'cost_aud', 'peak_power_kw',
        F.trim('connector_type').alias('connector_type'),
        F.trim('session_status').alias('session_status'),
        'payment_amount_aud', 'payment_gst', 'payment_mode',
        'payment_status', 'processed_at', 'cost_per_kwh', 'cost_reconciliation',
    )
)
write_fact(fact_session, 'FactChargingSession', 'session_id', ['session_year', 'session_month'])

# ── FactPayments ──────────────────────────────────────────────────────────────
fact_payments = meta(
    payments_raw
    .join(lk_customer, on='customer_id', how='left')
    .withColumn('payment_time_key', time_key(F.col('processed_at')))
    .withColumn('payment_year',  F.year('processed_at'))
    .withColumn('payment_month', F.month('processed_at'))
    .withColumn('refund_flag',
        F.when(F.lower('status').isin('refunded', 'reversed'), True).otherwise(False))
    .select(
        'payment_id', 'session_id', 'customer_key', 'customer_id',
        'payment_time_key', 'payment_year', 'payment_month',
        F.col('processed_at').alias('payment_ts'),
        'amount_aud', 'gst',
        F.trim('gateway').alias('gateway'),
        F.trim('payment_mode').alias('payment_mode'),
        F.trim('status').alias('payment_status'), 'refund_flag',
    )
)
write_fact(fact_payments, 'FactPayments', 'payment_id', ['payment_year', 'payment_month'])

# ── FactComplaints ────────────────────────────────────────────────────────────
fact_complaints = meta(
    complaints_raw
    .join(lk_customer, on='customer_id', how='left')
    .join(lk_station,  on='station_id',  how='left')
    .withColumn('complaint_time_key', time_key(F.col('created_ts')))
    .withColumn('complaint_year',  F.year('created_ts'))
    .withColumn('complaint_month', F.month('created_ts'))
    .withColumn('is_resolved',
        F.when(F.lower('status').isin('resolved', 'closed'), True).otherwise(False))
    .withColumn('resolution_days',
        F.when(F.col('resolved_ts').isNotNull() & F.col('created_ts').isNotNull(),
               ((F.col('resolved_ts').cast('long') - F.col('created_ts').cast('long')) / 86400
               ).cast('decimal(8,2)')))
    .select(
        'complaint_id', 'customer_key', 'customer_id', 'station_key', 'station_id',
        'complaint_time_key', 'complaint_year', 'complaint_month',
        F.col('created_ts').alias('complaint_created_ts'),
        F.col('resolved_ts').alias('complaint_resolved_ts'),
        F.trim('category').alias('complaint_category'),
        F.trim('priority').alias('priority'), 'sla_breach',
        F.trim('status').alias('complaint_status'), 'is_resolved', 'resolution_days',
    )
)
write_fact(fact_complaints, 'FactComplaints', 'complaint_id', ['complaint_year', 'complaint_month'])

# ── FactMaintenance ───────────────────────────────────────────────────────────
fact_maintenance = meta(
    maintenance_raw
    .join(lk_charger, on='charger_id', how='left')
    .join(lk_station, on='station_id', how='left')
    .join(
        lk_employee.withColumnRenamed('employee_id', '_emp_id'),
        maintenance_raw['employee_id'] == F.col('_emp_id'), how='left'
    ).drop('_emp_id')
    .withColumn('event_time_key', time_key(F.col('event_ts')))
    .withColumn('event_year',  F.year('event_ts'))
    .withColumn('event_month', F.month('event_ts'))
    .select(
        'event_id', 'charger_key', 'charger_id', 'station_key', 'station_id',
        'employee_key', 'employee_id', 'event_time_key', 'event_year', 'event_month',
        'event_ts', F.trim('root_cause').alias('root_cause'), 'mttr_hours',
    )
)
write_fact(fact_maintenance, 'FactMaintenance', 'event_id', ['event_year', 'event_month'])

# ── FactRealtimeSession ───────────────────────────────────────────────────────
fact_realtime = meta(
    rt_sessions
    .join(lk_charger,  on='charger_id',  how='left')
    .join(lk_station,  on='station_id',  how='left')
    .join(lk_customer, on='customer_id', how='left')
    .join(lk_vehicle,  on='vehicle_id',  how='left')
    .join(lk_tariff,   on='tariff_id',   how='left')
    .withColumn('session_time_key', time_key(F.col('plug_in_ts')))
    .withColumn('session_year',  F.year('plug_in_ts'))
    .withColumn('session_month', F.month('plug_in_ts'))
    .withColumn('power_efficiency_pct',
        F.when(F.col('peak_kw').isNotNull() & (F.col('peak_kw') > 0) &
               F.col('duration_min').isNotNull() & (F.col('duration_min') > 0),
               (F.col('energy_kwh') / (F.col('peak_kw') * F.col('duration_min') / 60) * 100
               ).cast('decimal(6,2)')))
    .select(
        'session_id', 'charger_key', 'charger_id', 'station_key', 'station_id',
        'customer_key', 'customer_id', 'vehicle_key', 'vehicle_id',
        'tariff_key', 'tariff_id', 'session_time_key', 'session_year', 'session_month',
        F.col('plug_in_ts').alias('session_start_ts'),
        F.col('charge_end_ts').alias('session_end_ts'),
        'duration_min', 'energy_kwh',
        F.col('peak_kw').alias('peak_power_kw'),
        F.trim('connector_type').alias('connector_type'),
        F.trim('session_status').alias('session_status'),
        F.col('raw_device_temp_c').alias('device_temp_c'),
        'signal_strength_dbm',
        F.col('firmware_ver').alias('firmware_version'),
        F.trim('state_code').alias('state_code'),
        F.trim('protocol_version').alias('protocol_version'),
        'power_efficiency_pct',
    )
)
write_fact(fact_realtime, 'FactRealtimeSession', 'session_id', ['session_year', 'session_month'])

In [ ]:
# ── FINAL SUMMARY ────────────────────────────────────────────────────────────
import json

ALL_TABLES = [
    (GOLD_DIM,  'DimState'),   (GOLD_DIM, 'DimCity'),    (GOLD_DIM, 'DimStation'),
    (GOLD_DIM,  'DimCharger'), (GOLD_DIM, 'DimCustomer'),(GOLD_DIM, 'DimVehicle'),
    (GOLD_DIM,  'DimEmployee'),(GOLD_DIM, 'DimPartner'), (GOLD_DIM, 'DimChargeCard'),
    (GOLD_DIM,  'DimTariff'),  (GOLD_DIM, 'DimTime'),
    (GOLD_FACT, 'FactChargingSession'), (GOLD_FACT, 'FactPayments'),
    (GOLD_FACT, 'FactComplaints'),      (GOLD_FACT, 'FactMaintenance'),
    (GOLD_FACT, 'FactRealtimeSession'),
]

summary = {'run_ts': RUN_TS, 'pipeline': PIPELINE, 'load_type': LOAD_TYPE, 'tables': []}

print('=' * 60)
print(f'GOLD LAYER SUMMARY  |  load_type={LOAD_TYPE}  |  {RUN_TS}')
print('=' * 60)
for base, name in ALL_TABLES:
    path = f'{base}/{name}'
    try:
        cnt = spark.read.format('delta').load(path).count()
        summary['tables'].append({'name': name, 'rows': cnt, 'status': 'ok'})
        print(f'  {name:<28} {cnt:>9} rows  OK')
    except Exception as e:
        summary['tables'].append({'name': name, 'rows': 0, 'status': 'missing'})
        print(f'  {name:<28} MISSING  ({e})')

total = sum(t['rows'] for t in summary['tables'])
print(f'\n  Total rows across all Gold tables: {total:,}')
print('=' * 60)

dbutils.notebook.exit(json.dumps(summary))